## 0. Setup (run first, every session)
**Before you start**
1. *Kaggle:* Settings -> Accelerator = **GPU T4/P100**, Internet = **On**; Add-ons -> Secrets -> add `HF_TOKEN` (a Hugging Face token with *write* access). *Colab:* Runtime -> GPU, and add `HF_TOKEN` under the key icon (Secrets), notebook access on.
2. Edit `HF_REPO` below (e.g. `yourname/atml-pa3-backup`). A **private** repo is created automatically on the first push.
3. A background loop pushes results + checkpoints to that repo every 10 minutes, and `pull` restores them in a fresh session. **If a session dies, re-run section 0 and then re-run the same training cell: it resumes from the last checkpoint.**

4. **Not yet run end to end** - these cells follow the same pattern as Task 1 but Task 2 code has only been syntax-checked. Run the smoke steps (2.2) first and expect to debug.
5. COCO 2014 is large (~19 GB of images). On Kaggle, attach a COCO-2014 dataset via *Add Input* and symlink it (cell 2.0); on Colab use `scripts.download_assets --coco`. VQAv2 files are small.

In [ ]:
import os, sys, subprocess, threading, time
HF_REPO  = "YOUR_USERNAME/atml-pa3-backup"      # <- edit
REPO_URL = "https://github.com/ZainFarhan-Work/ATML.git"
BRANCH   = "main"

ON_KAGGLE = os.path.exists("/kaggle")
ROOT = "/kaggle/working" if ON_KAGGLE else "/content"
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
else:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

if not os.path.exists(f"{ROOT}/ATML"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, f"{ROOT}/ATML"], check=True)
else:
    subprocess.run(["git", "-C", f"{ROOT}/ATML", "pull"], check=False)
os.chdir(f"{ROOT}/ATML/PA 3")
print(os.getcwd())

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m scripts.check_environment

In [ ]:
# restore anything saved by a previous session (harmless on the first run)
r = subprocess.run([sys.executable, "-m", "scripts.sync_hub", "pull", "--repo", HF_REPO, "--task", "task2"])
print("pull exit code:", r.returncode)

# background backup every 10 min (and call sync_now() manually after each section)
def sync_now():
    subprocess.run([sys.executable, "-m", "scripts.sync_hub", "push", "--repo", HF_REPO, "--task", "task2"])
def _loop():
    while True:
        time.sleep(600)
        try: sync_now()
        except Exception as e: print("sync failed:", e)
threading.Thread(target=_loop, daemon=True).start()

## 2.0 Data assets

In [ ]:
# Kaggle: attach COCO-2014 + VQAv2 as inputs, then link them here (adjust the input paths shown in the sidebar)
# !mkdir -p data/coco && ln -sfn /kaggle/input/<coco-dataset>/train2014 data/coco/train2014 && ln -sfn /kaggle/input/<coco-dataset>/val2014 data/coco/val2014
# Colab / anywhere with internet:
!python -m scripts.download_assets --vqa
# !python -m scripts.download_assets --coco

## 2.1 Build splits and cache SigLIP features (Step 1)

In [ ]:
!python -m task2_vlm.build_data
!python -m task2_vlm.cache_features
!python -m task2_vlm.checks
sync_now()

## 2.2 Smoke test (a few updates, then delete)

In [ ]:
!python -m task2_vlm.train --run-name smoke --max-updates 3
!rm -rf results/task2/smoke checkpoints/task2/smoke

## 2.3 Main run: caption alignment -> visual instruction tuning -> natural VQA (4,500 updates, resumable)

In [ ]:
!python -m task2_vlm.train --run-name main --resume
sync_now()

## 2.4 Final evaluation (4 image conditions) and analysis

In [ ]:
!python -m task2_vlm.evaluate --run-name main
!python -m task2_vlm.analyze --run-name main
sync_now()

## 2.5 (Optional) 49-token compression

In [ ]:
!python -m task2_vlm.train --run-name tokens49 --compress --resume
!python -m task2_vlm.evaluate --run-name tokens49 --compress
!python -m task2_vlm.analyze --run-name tokens49
sync_now()